# 案例：任务拆解与依赖排序

> Last updated: 2026-11-15

## 核心学习目标

本案例以多城市旅行规划请求为对象，学习如何根据任务分解阶段可使用的专业能力把复杂用户目标拆解为可执行的子任务，再根据各子任务之间以及能力输入、输出定义之间的数据关系分析依赖，最终形成带执行顺序的计划。

## 案例设计

案例按照“理解用户目标—加载能力目录—任务分解—依赖分析—执行排序”的流程组织。任务分解提示词要求模型保留城市、日期、预算和偏好等槽位；依赖分析提示词要求模型比较子任务输入与技能输出定义，并在不引入多余依赖的前提下生成执行顺序。

模型创建和调用统一位于 `llm.py`，本 Notebook 保留任务分解、结果解析、依赖分析和执行计划等教学逻辑。任务分解与依赖分析都通过真实大模型调用完成。

**学习重点**：
1. 理解专业能力描述如何约束任务分解边界。
2. 掌握把用户 Query 转换为带完整槽位子任务的方法。
3. 理解如何根据能力的技能输入、输出定义识别必要依赖。
4. 掌握把模型返回的 JSON 顺序校验为完整执行计划的方法。

### 模块与代码对应关系

| 环节 | 学习内容 | 对应实现 |
|------|----------|----------|
| 模型访问 | 环境加载、模型创建、文本提取与 JSON 解析 | `llm.py` |
| 任务分解 | 根据能力目录生成带槽位的子任务 | `task_decomposition_prompt_template`、`parse_decomposition_response` |
| 依赖分析 | 根据技能输入、输出定义生成执行顺序 | `dependency_analysis_system_prompt`、`invoke_json` |
| 执行计划 | 校验模型顺序、补全遗漏任务并输出最终计划 | `normalize_execution_order` |

## 0. 环境

本案例使用仓库根目录的共享 `.venv`。运行 Notebook 前，请在仓库根目录执行：

```bash
uv pip install -r Chapter-4/requirements.txt
```

环境变量统一从仓库根目录 `.env` 加载，不在章节目录维护局部 `.env`。运行前必须配置 `DASHSCOPE_API_KEY` 或 `OPENAI_API_KEY` 之一。

In [1]:
import sys
from datetime import datetime
from pathlib import Path

if sys.platform == "win32":
    for stream in (sys.stdout, sys.stderr):
        try:
            stream.reconfigure(encoding="utf-8")
        except Exception:
            pass

chapter4_candidate = Path.cwd()
if not (chapter4_candidate / "task_decomposition.py").exists():
    chapter4_candidate = Path.cwd() / "Chapter-4"

if not (chapter4_candidate / "task_decomposition.py").exists():
    raise FileNotFoundError("Could not locate Chapter-4/task_decomposition.py")

book_root_candidate = chapter4_candidate.parent
if not (book_root_candidate / "llm.py").exists():
    raise FileNotFoundError("Could not locate repository-root llm.py")

sys.path.insert(0, str(chapter4_candidate))
sys.path.insert(0, str(book_root_candidate))

from llm import ENV_FILE, PROJECT_ROOT, model_configured

print("Chapter-4:", chapter4_candidate)
print("项目根目录:", PROJECT_ROOT)
print("环境文件:", ENV_FILE)
print("模型凭据:", "已配置" if model_configured() else "未配置")
print("当前时间:", datetime.now().astimezone().strftime("%Y-%m-%d %H:%M:%S %z"))

Chapter-4: e:\Book\agent-systems-book\Chapter-4
项目根目录: E:\Book\agent-systems-book
环境文件: E:\Book\agent-systems-book\.env
模型凭据: 已配置
当前时间: 2026-11-15 17:46:40 +0800


## 1. 定义用户请求

本案例使用一个多城市旅行规划请求，用户目标中同时包含目的地、行程、酒店、天气和美食等约束。后续任务分解需要保留这些槽位。

In [2]:
# 可替换为你的测试请求
query = "你能帮我规划一个五一假期的多城市旅行吗？我还没想好行程顺序…… 大概是上海、苏州、杭州这几个地方？需要包含行程路线、酒店推荐、天气情况和美食攻略"

## 2. 初始化模型客户端

模型配置和所有模型调用都封装在 `llm.py`。本单元直接创建模型客户端；后续任务分解和依赖分析都会调用真实模型。缺少模型凭据时，`create_llm()` 会给出明确错误。

In [3]:
from llm import create_llm, invoke_chat, invoke_json

llm = create_llm()
print(f"✓ 模型客户端已初始化: {llm.model_name}")

✓ 模型客户端已初始化: qwen3.7-plus


## 3. 定义任务分解能力目录

本案例中的能力目录是提供给任务拆解大模型的**可用能力清单**。它列出当前可使用的专业能力、每项能力适合完成什么任务，以及相关技能的输入与输出定义。

任务拆解器本身不会执行子任务，它依据能力目录判断用户目标能否被现有能力覆盖，再把目标分解为多个可执行子任务。每个条目包含：

- `name` 与 `description`：说明团队中存在哪类专业能力，用于任务分解。
- `skills.inputSchema`：说明完成技能需要哪些信息，用于检查槽位是否完整。
- `skills.outputSchema`：说明技能会产出哪些结果，用于依赖分析。

因此，本章可以把能力目录理解为“任务规划阶段可以使用的能力清单”。这里重点讨论如何借助这份清单完成规划，而不展开智能体实例如何创建、如何路由、如何互相通信以及如何并行执行。

全书案例最终是一套完整的多智能体编排系统：第 6 章会说明这些能力如何由具体子智能体承担并形成不同编排模式。

In [4]:
from typing import Any

CAPABILITY_CATALOG = [
    {
        "name": "weather_forecast",
        "description": "查询指定城市、日期的天气预报。",
        "skills": [
            {
                "name": "get_weather",
                "inputSchema": ["city", "date"],
                "outputSchema": ["weather_summary", "temperature"],
            }
        ],
    },
    {
        "name": "attraction_recommendation",
        "description": "根据城市和偏好推荐旅游景点。",
        "skills": [
            {
                "name": "recommend_attractions",
                "inputSchema": ["city", "preferences"],
                "outputSchema": ["attraction_list"],
            }
        ],
    },
    {
        "name": "hotel_recommendation",
        "description": "根据城市、位置偏好和预算推荐酒店住宿。",
        "skills": [
            {
                "name": "recommend_hotels",
                "inputSchema": ["city", "location_preference", "budget_per_night"],
                "outputSchema": ["hotel_list"],
            }
        ],
    },
    {
        "name": "restaurant_recommendation",
        "description": "根据城市、菜系偏好和人均预算推荐当地特色餐厅。",
        "skills": [
            {
                "name": "recommend_restaurants",
                "inputSchema": ["city", "cuisine_preference", "budget_per_person"],
                "outputSchema": ["restaurant_list"],
            }
        ],
    },
    {
        "name": "flight_search",
        "description": "查询出发地到目的地的航班信息。",
        "skills": [
            {
                "name": "search_flights",
                "inputSchema": ["origin", "destination", "date"],
                "outputSchema": ["flight_list"],
            }
        ],
    },
    {
        "name": "itinerary_planning",
        "description": "根据天数、天气和景点列表生成行程安排。",
        "skills": [
            {
                "name": "plan_itinerary",
                "inputSchema": [
                    "city_route",
                    "days",
                    "dates",
                    "weather_summary",
                    "attraction_list",
                    "hotel_list",
                    "restaurant_list",
                ],
                "outputSchema": ["itinerary"],
            }
        ],
    },
]


def build_capability_catalog_text(capabilities: list[dict]) -> str:
    """Render capability descriptions for task decomposition."""
    team = ""
    for capability in capabilities:
        team += f"{capability['name']}：{capability['description']}\n    "
    return team


def build_capability_parameter_text(capabilities: list[dict]) -> str:
    """Render skill input/output definitions for dependency analysis."""
    text = ""
    for capability in capabilities:
        name = capability.get("name", "")
        text += f"{name}\n"
        for skill in capability.get("skills", []):
            text += (
                f"\t{skill['name']}, "
                f"inputSchema:{skill.get('inputSchema', [])}, "
                f"outputSchema:{skill.get('outputSchema', [])}\n"
            )
    return text


capability_catalog: list[dict[str, Any]] = CAPABILITY_CATALOG
# Natural-language capability view used by the task decomposition prompt.
capability_catalog_text = build_capability_catalog_text(capability_catalog)
# Structured input/output view used by the dependency analysis prompt.
capability_parameter_text = build_capability_parameter_text(capability_catalog)
print("--- 能力目录（任务分解用）---")
print(
    capability_catalog_text[:800]
    + ("..." if len(capability_catalog_text) > 800 else "")
)

--- 能力目录（任务分解用）---
weather_forecast：查询指定城市、日期的天气预报。
    attraction_recommendation：根据城市和偏好推荐旅游景点。
    hotel_recommendation：根据城市、位置偏好和预算推荐酒店住宿。
    restaurant_recommendation：根据城市、菜系偏好和人均预算推荐当地特色餐厅。
    flight_search：查询出发地到目的地的航班信息。
    itinerary_planning：根据天数、天气和景点列表生成行程安排。
    


## 4. 第一步：任务分解提示词

提示词中的 `{}` 依次填入**背景信息**、**能力目录**和**用户输入**。这里将能力目录作为任务拆解的约束边界：分解结果只能使用目录中已经声明的能力，不能凭空增加目录之外的专业能力。后续依赖分析还会使用同一份目录中的技能输入、输出定义。

In [5]:
task_decomposition_prompt_template = """
##角色：你是一个高度熟练的任务拆解专家，专注于将用户输入拆解为粒度适当的子任务。
##任务：根据用户的输入，分析目标以及能力目录中所有可用能力的描述，通过将用户输入分解为能力目录可以覆盖的信息明确的子任务，以帮助有效地实现用户的目标。
##背景信息（关于具体任务拆解所需要知道的相关信息及要求）：
    {}
##可用能力目录（可用于完成子任务的候选能力）：
    {}
##步骤：
    1. **理解目标**：根据所有用户输入确定用户想要实现的核心目标和结果。
    2. **考虑背景信息**：提供的背景信息中包含拆解任务所需要的额外的领域知识，可用作参考。
    3. **定义子任务**：根据目标的复杂程度将任务拆解为子任务。
      3.1 **任务拆解原则**：
        - 简单：每个子任务都是一个能够匹配能力目录中单一能力的祈使句指令。
        - 信息完整：每个子任务指令中必须包含明确的槽位信息，所需槽位信息从用户输入及目标中抽取。
        - 优化复杂度：根据任务复杂度调整子任务数量，使用最少子任务完成整体目标。
      3.2 **子任务格式**：不同子任务间使用#分割。
    4. **确认子任务能否完成**：逐个判断每个子任务能否由能力目录中的某项能力完成；只有某个子任务确实无法覆盖时，才将该子任务标记为'NULL'，能覆盖的子任务必须保留，不得把全部子任务统一标记为'NULL'。
    5. **补充子任务信息**：针对每个子任务，提取该子任务的所有槽位，并根据目标补全所有槽位，合并成一句完整的祈使指令生成新的子任务并替换原有子任务，使子任务内容信息明确指令清晰。
##要求：
    1.需要根据完整的用户输入的语义（包含所有输入内容）进行分析；
    2.背景信息仅作为参考，只有当需要时才使用；
    3.子任务仅能依靠能力目录中声明的技能完成，避免擅自扩展为目录之外的能力并创造无法完成的子任务。
    4.仅输出简洁且信息完整的整体目标内容以及槽位信息完整的任务拆解结果，不要输出分析过程及原因;
    5.子任务中必须包含明确完整的槽位信息，避免直接将智能体能力作为子任务输出；
    6.输出的子任务应为信息明确的完整指令，确保用户输入及目标中包含的所有信息都出现在对应在子任务中，避免槽位信息丢失或单独输出槽位信息作为子任务；
    7.能力目录中的不同能力可以组合完成复杂目标。某一个子任务不可覆盖时，只处理该子任务，不能用'NULL'否定其他可完成子任务。
##输出格式：
    # 目标
        整体目标内容（不包含人称的祈使句式）
    # 任务拆解
     - 子任务1
     - 子任务2(如有)
     - ...
##用户输入：
    {}
"""

background_info = ""
task_decomposition_prompt = task_decomposition_prompt_template.format(
    background_info,
    capability_catalog_text,
    query,
)
print(f"任务分解提示词字符数: {len(task_decomposition_prompt)}")

任务分解提示词字符数: 1524


## 5. 展示：拼好后的「任务分解」完整提示词

下面就是第一步要发给模型的 **user 消息**。

In [6]:
from IPython.display import Markdown, display

display(Markdown(f"**字符数：** {len(task_decomposition_prompt)}"))
display(Markdown("```text\n" + task_decomposition_prompt[:2500] + ("\n...(截断)..." if len(task_decomposition_prompt) > 2500 else "") + "\n```"))

**字符数：** 1524

```text

##角色：你是一个高度熟练的任务拆解专家，专注于将用户输入拆解为粒度适当的子任务。
##任务：根据用户的输入，分析目标以及能力目录中所有可用能力的描述，通过将用户输入分解为能力目录可以覆盖的信息明确的子任务，以帮助有效地实现用户的目标。
##背景信息（关于具体任务拆解所需要知道的相关信息及要求）：
    
##可用能力目录（可用于完成子任务的候选能力）：
    weather_forecast：查询指定城市、日期的天气预报。
    attraction_recommendation：根据城市和偏好推荐旅游景点。
    hotel_recommendation：根据城市、位置偏好和预算推荐酒店住宿。
    restaurant_recommendation：根据城市、菜系偏好和人均预算推荐当地特色餐厅。
    flight_search：查询出发地到目的地的航班信息。
    itinerary_planning：根据天数、天气和景点列表生成行程安排。
    
##步骤：
    1. **理解目标**：根据所有用户输入确定用户想要实现的核心目标和结果。
    2. **考虑背景信息**：提供的背景信息中包含拆解任务所需要的额外的领域知识，可用作参考。
    3. **定义子任务**：根据目标的复杂程度将任务拆解为子任务。
      3.1 **任务拆解原则**：
        - 简单：每个子任务都是一个能够匹配能力目录中单一能力的祈使句指令。
        - 信息完整：每个子任务指令中必须包含明确的槽位信息，所需槽位信息从用户输入及目标中抽取。
        - 优化复杂度：根据任务复杂度调整子任务数量，使用最少子任务完成整体目标。
      3.2 **子任务格式**：不同子任务间使用#分割。
    4. **确认子任务能否完成**：逐个判断每个子任务能否由能力目录中的某项能力完成；只有某个子任务确实无法覆盖时，才将该子任务标记为'NULL'，能覆盖的子任务必须保留，不得把全部子任务统一标记为'NULL'。
    5. **补充子任务信息**：针对每个子任务，提取该子任务的所有槽位，并根据目标补全所有槽位，合并成一句完整的祈使指令生成新的子任务并替换原有子任务，使子任务内容信息明确指令清晰。
##要求：
    1.需要根据完整的用户输入的语义（包含所有输入内容）进行分析；
    2.背景信息仅作为参考，只有当需要时才使用；
    3.子任务仅能依靠能力目录中声明的技能完成，避免擅自扩展为目录之外的能力并创造无法完成的子任务。
    4.仅输出简洁且信息完整的整体目标内容以及槽位信息完整的任务拆解结果，不要输出分析过程及原因;
    5.子任务中必须包含明确完整的槽位信息，避免直接将智能体能力作为子任务输出；
    6.输出的子任务应为信息明确的完整指令，确保用户输入及目标中包含的所有信息都出现在对应在子任务中，避免槽位信息丢失或单独输出槽位信息作为子任务；
    7.能力目录中的不同能力可以组合完成复杂目标。某一个子任务不可覆盖时，只处理该子任务，不能用'NULL'否定其他可完成子任务。
##输出格式：
    # 目标
        整体目标内容（不包含人称的祈使句式）
    # 任务拆解
     - 子任务1
     - 子任务2(如有)
     - ...
##用户输入：
    你能帮我规划一个五一假期的多城市旅行吗？我还没想好行程顺序…… 大概是上海、苏州、杭州这几个地方？需要包含行程路线、酒店推荐、天气情况和美食攻略

```

## 6. 调用 LLM 做任务分解

本单元使用能力目录和用户请求构造提示词，并调用真实模型生成任务分解结果。模型调用或输出解析失败时直接暴露错误，便于定位提示词、模型或环境配置问题。

In [7]:
task_decomposition_response = await invoke_chat(
    llm,
    task_decomposition_prompt,
    system_prompt="You are a helpful assistant.",
)
print("✓ 已调用 LLM 完成任务分解")

from IPython.display import Markdown, display

display(Markdown("### 模型输出（任务分解）"))
display(Markdown("```text\n" + task_decomposition_response + "\n```"))

✓ 已调用 LLM 完成任务分解


### 模型输出（任务分解）

```text
# 目标
规划五一假期上海、苏州、杭州的多城市旅行行程，包含路线、酒店、天气及美食推荐

# 任务拆解
 - 查询上海五一假期的天气预报
 - 查询苏州五一假期的天气预报
 - 查询杭州五一假期的天气预报
 - 根据上海城市特点推荐旅游景点
 - 根据苏州城市特点推荐旅游景点
 - 根据杭州城市特点推荐旅游景点
 - 根据上海位置偏好和预算推荐酒店住宿
 - 根据苏州位置偏好和预算推荐酒店住宿
 - 根据杭州位置偏好和预算推荐酒店住宿
 - 根据上海菜系偏好和人均预算推荐当地特色餐厅
 - 根据苏州菜系偏好和人均预算推荐当地特色餐厅
 - 根据杭州菜系偏好和人均预算推荐当地特色餐厅
 - 查询出发地到上海的航班信息
 - 查询上海到苏州的交通方式（注：能力目录无城际交通查询，标记为NULL或忽略，此处严格遵循目录，若无对应能力则无法生成子任务，但考虑到 itinerary_planning 可能隐含路线，先提取明确能力。由于目录中只有 flight_search，且苏杭之间多为高铁，flight_search 不适用，故仅保留航班相关或依赖 itinerary_planning 生成整体路线）
 - 根据天数、天气和上海、苏州、杭州的景点列表生成行程安排
```

## 7. 解析任务分解结果

从 `# 目标` / `# 任务拆解` 段落提取 **totalGoal** 与 **subSteps** 列表。

In [8]:
import re


def parse_decomposition_response(response: str) -> dict:
    """Parse total goal and subtasks from the model response."""
    totalgoal_key = "# 目标"
    substep_key = "# 任务拆解"

    sub_steps = []
    goal_lines = []
    reached_tasks = False

    for line in response.splitlines():
        line = line.strip()
        if not reached_tasks:
            if line.startswith(substep_key):
                reached_tasks = True
                continue
            if line.startswith(totalgoal_key):
                continue
            if line:
                goal_lines.append(line)
        elif line.startswith(("-", "*")):
            task = re.sub(r"^[-*]\s+", "", line).strip()
            if task:
                sub_steps.append(task)

    if not sub_steps:
        sub_steps = ["NULL"]
    return {"totalGoal": " ".join(goal_lines), "subSteps": sub_steps}


decomposition = parse_decomposition_response(task_decomposition_response)
print(f"总体目标: {decomposition['totalGoal']}")
print(f"子任务 ({len(decomposition['subSteps'])}):")
for i, step in enumerate(decomposition['subSteps'], 1):
    print(f"  {i}. {step}")

总体目标: 规划五一假期上海、苏州、杭州的多城市旅行行程，包含路线、酒店、天气及美食推荐
子任务 (15):
  1. 查询上海五一假期的天气预报
  2. 查询苏州五一假期的天气预报
  3. 查询杭州五一假期的天气预报
  4. 根据上海城市特点推荐旅游景点
  5. 根据苏州城市特点推荐旅游景点
  6. 根据杭州城市特点推荐旅游景点
  7. 根据上海位置偏好和预算推荐酒店住宿
  8. 根据苏州位置偏好和预算推荐酒店住宿
  9. 根据杭州位置偏好和预算推荐酒店住宿
  10. 根据上海菜系偏好和人均预算推荐当地特色餐厅
  11. 根据苏州菜系偏好和人均预算推荐当地特色餐厅
  12. 根据杭州菜系偏好和人均预算推荐当地特色餐厅
  13. 查询出发地到上海的航班信息
  14. 查询上海到苏州的交通方式（注：能力目录无城际交通查询，标记为NULL或忽略，此处严格遵循目录，若无对应能力则无法生成子任务，但考虑到 itinerary_planning 可能隐含路线，先提取明确能力。由于目录中只有 flight_search，且苏杭之间多为高铁，flight_search 不适用，故仅保留航班相关或依赖 itinerary_planning 生成整体路线）
  15. 根据天数、天气和上海、苏州、杭州的景点列表生成行程安排


## 8. 第二步：依赖分析提示词

为每个子任务分配临时 ID（T1、T2…），把 **subtasks** 与**能力目录中的技能参数**填入 user prompt；system prompt 规定 JSON 排序输出格式。

In [9]:
# 依赖分析提示词保留在 Notebook 中，后续迁移到 Chapter-6 时抽取为独立模块。
dependency_analysis_system_prompt = """
你是一个任务规划系统中的[任务依赖分析器], 负责根据子任务与候选能力的技能输入输出定义, 识别子任务之间的执行依赖后调整子任务的执行顺序

# 输入
1. subtasks: 已拆分的子任务列表, 每个子任务包含:
   - task_id（如 T1、T2 等）
   - task_description（子任务的自然语言描述）
2. capabilities: 候选能力列表, 每个能力提供若干技能, 每个技能包含:
   - skill_name
   - input_parameters
   - output_parameters

# 目标
你的目标是分析子任务之间的必要依赖关系后调整子任务的执行顺序
第一步：建立全局输入输出视图  
- 遍历所有子任务  
- 对每个子任务，判断最可能由哪项能力的哪个技能来完成  
- 读取并理解该技能的 输入参数 和 输出参数  
- 记录每个子任务的 [输入] 和 [输出]  

第二步：识别必要的任务依赖  
- 在获得所有子任务的输入输出信息后，从全局视角分析依赖关系 
- 如果子任务 A 输入参数中要求的信息，能够从子任务B的输出参数中获取，则A依赖B， B要先于A执行 
- 不要引入不确定或非必要的依赖关系

第三步：调整执行顺序  
- 仅当存在明确且必要的依赖关系时，才调整子任务顺序, 对不存在依赖关系的子任务，保持其原有的相对顺序  
- 调整后的顺序必须满足所有必要的依赖约束  
- 不要进行多余的重排

# 输出
- 输出必须是合法的 JSON。
- 仅输出调整后的子任务执行顺序, 使用如下格式:
{
  "1": "task_id",
  "2": "task_id",
  ...
}
其中order是执行顺序，从1开始递增; task_id是输入部分的子任务id, 必须与输入中的子任务对应
"""

dependency_analysis_user_prompt_template = """
subtasks:
{subtasks}

capabilities:
{capabilities}

请严格按照输出格式返回, 除此之外不要有其他输出
"""

sub_steps = decomposition["subSteps"]
id_to_task = {f"T{i + 1}": task for i, task in enumerate(sub_steps)}

dependency_analysis_user_prompt = dependency_analysis_user_prompt_template.format(
    subtasks=id_to_task,
    capabilities=capability_parameter_text,
)

print("子任务 ID 映射:")
for tid, desc in id_to_task.items():
    print(f"  {tid}: {desc}")
print(f"\n依赖分析用户提示词字符数: {len(dependency_analysis_user_prompt)}")

子任务 ID 映射:
  T1: 查询上海五一假期的天气预报
  T2: 查询苏州五一假期的天气预报
  T3: 查询杭州五一假期的天气预报
  T4: 根据上海城市特点推荐旅游景点
  T5: 根据苏州城市特点推荐旅游景点
  T6: 根据杭州城市特点推荐旅游景点
  T7: 根据上海位置偏好和预算推荐酒店住宿
  T8: 根据苏州位置偏好和预算推荐酒店住宿
  T9: 根据杭州位置偏好和预算推荐酒店住宿
  T10: 根据上海菜系偏好和人均预算推荐当地特色餐厅
  T11: 根据苏州菜系偏好和人均预算推荐当地特色餐厅
  T12: 根据杭州菜系偏好和人均预算推荐当地特色餐厅
  T13: 查询出发地到上海的航班信息
  T14: 查询上海到苏州的交通方式（注：能力目录无城际交通查询，标记为NULL或忽略，此处严格遵循目录，若无对应能力则无法生成子任务，但考虑到 itinerary_planning 可能隐含路线，先提取明确能力。由于目录中只有 flight_search，且苏杭之间多为高铁，flight_search 不适用，故仅保留航班相关或依赖 itinerary_planning 生成整体路线）
  T15: 根据天数、天气和上海、苏州、杭州的景点列表生成行程安排

依赖分析用户提示词字符数: 1430


## 9. 展示：依赖分析完整提示词

In [10]:
display(Markdown("### System Prompt"))
display(Markdown("```text\n" + dependency_analysis_system_prompt.strip() + "\n```"))
display(Markdown("### User Prompt"))
display(Markdown("```text\n" + dependency_analysis_user_prompt + "\n```"))

### System Prompt

```text
你是一个任务规划系统中的[任务依赖分析器], 负责根据子任务与候选能力的技能输入输出定义, 识别子任务之间的执行依赖后调整子任务的执行顺序

# 输入
1. subtasks: 已拆分的子任务列表, 每个子任务包含:
   - task_id（如 T1、T2 等）
   - task_description（子任务的自然语言描述）
2. capabilities: 候选能力列表, 每个能力提供若干技能, 每个技能包含:
   - skill_name
   - input_parameters
   - output_parameters

# 目标
你的目标是分析子任务之间的必要依赖关系后调整子任务的执行顺序
第一步：建立全局输入输出视图  
- 遍历所有子任务  
- 对每个子任务，判断最可能由哪项能力的哪个技能来完成  
- 读取并理解该技能的 输入参数 和 输出参数  
- 记录每个子任务的 [输入] 和 [输出]  

第二步：识别必要的任务依赖  
- 在获得所有子任务的输入输出信息后，从全局视角分析依赖关系 
- 如果子任务 A 输入参数中要求的信息，能够从子任务B的输出参数中获取，则A依赖B， B要先于A执行 
- 不要引入不确定或非必要的依赖关系

第三步：调整执行顺序  
- 仅当存在明确且必要的依赖关系时，才调整子任务顺序, 对不存在依赖关系的子任务，保持其原有的相对顺序  
- 调整后的顺序必须满足所有必要的依赖约束  
- 不要进行多余的重排

# 输出
- 输出必须是合法的 JSON。
- 仅输出调整后的子任务执行顺序, 使用如下格式:
{
  "1": "task_id",
  "2": "task_id",
  ...
}
其中order是执行顺序，从1开始递增; task_id是输入部分的子任务id, 必须与输入中的子任务对应
```

### User Prompt

```text

subtasks:
{'T1': '查询上海五一假期的天气预报', 'T2': '查询苏州五一假期的天气预报', 'T3': '查询杭州五一假期的天气预报', 'T4': '根据上海城市特点推荐旅游景点', 'T5': '根据苏州城市特点推荐旅游景点', 'T6': '根据杭州城市特点推荐旅游景点', 'T7': '根据上海位置偏好和预算推荐酒店住宿', 'T8': '根据苏州位置偏好和预算推荐酒店住宿', 'T9': '根据杭州位置偏好和预算推荐酒店住宿', 'T10': '根据上海菜系偏好和人均预算推荐当地特色餐厅', 'T11': '根据苏州菜系偏好和人均预算推荐当地特色餐厅', 'T12': '根据杭州菜系偏好和人均预算推荐当地特色餐厅', 'T13': '查询出发地到上海的航班信息', 'T14': '查询上海到苏州的交通方式（注：能力目录无城际交通查询，标记为NULL或忽略，此处严格遵循目录，若无对应能力则无法生成子任务，但考虑到 itinerary_planning 可能隐含路线，先提取明确能力。由于目录中只有 flight_search，且苏杭之间多为高铁，flight_search 不适用，故仅保留航班相关或依赖 itinerary_planning 生成整体路线）', 'T15': '根据天数、天气和上海、苏州、杭州的景点列表生成行程安排'}

capabilities:
weather_forecast
	get_weather, inputSchema:['city', 'date'], outputSchema:['weather_summary', 'temperature']
attraction_recommendation
	recommend_attractions, inputSchema:['city', 'preferences'], outputSchema:['attraction_list']
hotel_recommendation
	recommend_hotels, inputSchema:['city', 'location_preference', 'budget_per_night'], outputSchema:['hotel_list']
restaurant_recommendation
	recommend_restaurants, inputSchema:['city', 'cuisine_preference', 'budget_per_person'], outputSchema:['restaurant_list']
flight_search
	search_flights, inputSchema:['origin', 'destination', 'date'], outputSchema:['flight_list']
itinerary_planning
	plan_itinerary, inputSchema:['city_route', 'days', 'dates', 'weather_summary', 'attraction_list', 'hotel_list', 'restaurant_list'], outputSchema:['itinerary']


请严格按照输出格式返回, 除此之外不要有其他输出

```

## 10. 调用 LLM 做依赖排序

期望返回 JSON，例如 `{"1": "T2", "2": "T1", "3": "T3"}`。

In [11]:
import json

def normalize_execution_order(order: object, task_map: dict[str, str]) -> list[str]:
    """Convert model output into a complete execution order."""
    ordered_ids: list[str] = []
    if isinstance(order, dict):
        for value in order.values():
            tid = str(value).strip()
            if tid in task_map and tid not in ordered_ids:
                ordered_ids.append(tid)
    for tid in task_map:
        if tid not in ordered_ids:
            ordered_ids.append(tid)
    return [task_map[tid] for tid in ordered_ids]


if len(sub_steps) < 2:
    order = None
    sorted_steps = sub_steps
    print("⚠ 子任务少于 2 个，跳过排序")
else:
    order = await invoke_json(
        llm,
        dependency_analysis_user_prompt,
        system_prompt=dependency_analysis_system_prompt,
    )
    print("✓ 已调用 LLM 完成依赖排序")
    print(json.dumps(order, ensure_ascii=False, indent=2))
    sorted_steps = normalize_execution_order(order, id_to_task)

✓ 已调用 LLM 完成依赖排序
{
  "1": "T1",
  "2": "T2",
  "3": "T3",
  "4": "T4",
  "5": "T5",
  "6": "T6",
  "7": "T7",
  "8": "T8",
  "9": "T9",
  "10": "T10",
  "11": "T11",
  "12": "T12",
  "13": "T13",
  "14": "T14",
  "15": "T15"
}


## 11. 最终执行计划

In [12]:
result = {
    "total_goal": decomposition["totalGoal"],
    "original_steps": sub_steps,
    "execution_order": order,
    "sorted_steps": sorted_steps,
}

print("=" * 60)
print("📊 最终执行计划")
print("=" * 60)
print(f"\n总体目标: {result['total_goal']}")
print(f"\n原始子任务顺序:")
for i, step in enumerate(result["original_steps"], 1):
    print(f"  {i}. {step}")
print(f"\n优化后执行顺序:")
for i, step in enumerate(result["sorted_steps"], 1):
    print(f"  {i}. {step}")

📊 最终执行计划

总体目标: 规划五一假期上海、苏州、杭州的多城市旅行行程，包含路线、酒店、天气及美食推荐

原始子任务顺序:
  1. 查询上海五一假期的天气预报
  2. 查询苏州五一假期的天气预报
  3. 查询杭州五一假期的天气预报
  4. 根据上海城市特点推荐旅游景点
  5. 根据苏州城市特点推荐旅游景点
  6. 根据杭州城市特点推荐旅游景点
  7. 根据上海位置偏好和预算推荐酒店住宿
  8. 根据苏州位置偏好和预算推荐酒店住宿
  9. 根据杭州位置偏好和预算推荐酒店住宿
  10. 根据上海菜系偏好和人均预算推荐当地特色餐厅
  11. 根据苏州菜系偏好和人均预算推荐当地特色餐厅
  12. 根据杭州菜系偏好和人均预算推荐当地特色餐厅
  13. 查询出发地到上海的航班信息
  14. 查询上海到苏州的交通方式（注：能力目录无城际交通查询，标记为NULL或忽略，此处严格遵循目录，若无对应能力则无法生成子任务，但考虑到 itinerary_planning 可能隐含路线，先提取明确能力。由于目录中只有 flight_search，且苏杭之间多为高铁，flight_search 不适用，故仅保留航班相关或依赖 itinerary_planning 生成整体路线）
  15. 根据天数、天气和上海、苏州、杭州的景点列表生成行程安排

优化后执行顺序:
  1. 查询上海五一假期的天气预报
  2. 查询苏州五一假期的天气预报
  3. 查询杭州五一假期的天气预报
  4. 根据上海城市特点推荐旅游景点
  5. 根据苏州城市特点推荐旅游景点
  6. 根据杭州城市特点推荐旅游景点
  7. 根据上海位置偏好和预算推荐酒店住宿
  8. 根据苏州位置偏好和预算推荐酒店住宿
  9. 根据杭州位置偏好和预算推荐酒店住宿
  10. 根据上海菜系偏好和人均预算推荐当地特色餐厅
  11. 根据苏州菜系偏好和人均预算推荐当地特色餐厅
  12. 根据杭州菜系偏好和人均预算推荐当地特色餐厅
  13. 查询出发地到上海的航班信息
  14. 查询上海到苏州的交通方式（注：能力目录无城际交通查询，标记为NULL或忽略，此处严格遵循目录，若无对应能力则无法生成子任务，但考虑到 itinerary_planning 可能隐含路线，先提取明确能力。由于目录中只有 

## 12. 小结

- **任务分解**：用户 Query + 能力目录 → 带槽位的子任务列表（`# 目标` / `# 任务拆解`）。
- **依赖排序**：子任务 ID + 能力目录中的技能输入输出 → LLM 输出 JSON 执行顺序 → 重排子任务。
- **代码边界**：Notebook 负责教学流程；`llm.py` 负责模型配置、调用和响应解析。后续接入第 6 章时，可把拆解与依赖逻辑分别抽成独立 Python 模块。
- **能力升级**：本章的能力目录用于任务分解与依赖分析；进入第 6 章后，会升级为多智能体系统中的领域智能体能力，并由具体领域智能体承载和执行。